In [1]:
import sys
sys.path.insert(0, '../lib')

In [2]:
import os
import pathlib
import typing
import datetime

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
import decoupler
import statsmodels.stats.multitest
import scanpy as sc

import common_data

/projects/b1196/envs/serniczek/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
pd.options.display.max_columns = 200
pd.options.display.max_rows = 200
%config InlineBackend.figure_format = "retina"

# Create pseudobulks

Create pseudobulks for all samples, removing transcripts from accession numbers only.

Requires at least 50 cells per sample to create a pseudobulk for a given cell type-sample pair.

In [4]:
def sanitize_name(name):
    return name.replace(' ', '_').replace('*', '').replace(';', '_and').replace('/', '_')

In [5]:
ROOT = common_data.DATA

In [ ]:
BASE = ROOT / '05_pseudobulk/01_all_pseudobulk'

In [7]:
os.makedirs(BASE, exist_ok=True)

In [8]:
adata = sc.read_h5ad(common_data.SC_RAW)

In [9]:
adata.shape

(2452841, 19488)

In [10]:
sc_labels = pd.read_csv(common_data.SC_LABELS, index_col=0)

In [11]:
PSEUDOBULK_CUTOFF = 50

In [12]:
METADATA_FIELDS = {
    'individual': 'sample',
    'Level_6': 'cell_type',
    'Sex_genes': 'sex',
}

In [13]:
class CellTypeInfo:
    def __init__(self):
        self.genes = []
        self.pseudobulks = []
        self.n_cells = []
        self.metadata = []
        self.sample_info = []

In [14]:
def create_pseudobulks(
    adata: sc.AnnData,
    samples: typing.Dict[str, typing.Collection[str]]
) -> typing.Dict[str, CellTypeInfo]:
    infos = {}
    for cell_type in adata.obs.Level_6.unique():
        info = CellTypeInfo()
        infos[cell_type] = info
        for label, sample_ids in samples.items():
            for sample in sample_ids:
                sample_idx = adata.obs.Level_6.eq(cell_type) & adata.obs.individual.eq(sample)
                sample_row = adata.obs.loc[
                    sample_idx,
                    list(METADATA_FIELDS.keys())
                ].head(1).rename(columns=METADATA_FIELDS)
                sample_row['n_cells'] = sample_idx.sum()
                info.sample_info.append(sample_row)
                if sample_idx.sum() < PSEUDOBULK_CUTOFF:
                    continue
                info.pseudobulks.append(adata.X[sample_idx].sum(axis=0).A1)
                info.metadata.append(sample_row)
    return infos

In [15]:
# Either starts with all this, or ends with -AS1/2/3 or -DT
FUNKY_TRANSCRIPTS = (
    '(^(RP\d{1,2}-|LINC|CT[ABCD]-|AC\d{6}|AL\d{6}'
    '|AP\d{6}|AF\d{6}|XXba|XXya|FO\d{6}|FP\d{6}).+|.+(-AS[123]?)$|.+-DT$)'
)

In [16]:
global_gene_idx = (
    ~adata.var.index.str.startswith("MT-")
    & ~adata.var_names.str.startswith('SARS-CoV-2')
    & ~adata.var.index.str.match(FUNKY_TRANSCRIPTS)
)

In [17]:
samples = list(adata.obs.individual.unique())

In [18]:
%%time
pseudobulks = create_pseudobulks(adata, {'all': samples})

CPU times: user 49.6 s, sys: 243 ms, total: 49.8 s
Wall time: 50 s


In [19]:
def save_info(
    cell_type: str,
    info: CellTypeInfo,
    global_gene_idx: typing.Collection[bool],
    genes: typing.Collection[str]
):
    ct_clean = sanitize_name(cell_type)
    path = BASE / ct_clean
    # Some cell type have ZERO samples
    if not info.metadata:
        return

    metadata = pd.concat(info.metadata).reset_index(drop=True)

    gene_names = genes[global_gene_idx]
    pseudobulk = np.vstack(info.pseudobulks)[:, global_gene_idx]
    pseudobulk = pd.DataFrame(pseudobulk, columns=gene_names, index=metadata['sample']).T

    os.makedirs(path, exist_ok=True)
    pseudobulk.to_csv(path / 'count.txt', sep='\t')
    metadata.to_csv(path / 'meta.csv')

In [20]:
for ct, info in pseudobulks.items():
    save_info(ct, info, global_gene_idx, adata.var_names)